In [10]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: e:\Data_Science_Bio\single-cell-pbmc-analysis\notebooks


### 1. Configuration & File Paths

In [ ]:
import scanpy as sc

# Fetching the filtered input file from previous step (01_data_ingestion.ipynb)
# Filtered for cells with < 2500 genes (removes doublets) and
# Cells with < 5% MT genes (removes dead cells)

input_file = "../data/processed/pbmc3k_qc_filtered.h5ad" #../ moves up one folder
output_file = "../data/processed/pca_computed.h5ad"

# feature selecting 2000 most variable genes and normalizing to 10,000 reads per cell
# These parameters are commonly used in single-cell RNA-seq analysis to focus on the most informative 
# genes and to account for differences in sequencing depth across cells.

# `n_top_genes` specifies the number of highly variable genes to select for downstream analysis, 
# while `target_reads` is used for normalizing the data to a common scale, allowing for more 
# accurate comparisons between cells.

n_top_genes = 2000
target_reads = 1e4

### 2. Execution

In [12]:
print(f"Loading filtered data from {input_file}...")
adata = sc.read_h5ad(input_file)

Loading filtered data from ../data/processed/pbmc3k_qc_filtered.h5ad...


In [ ]:
# Normalize and Log Transform

# Normalizing total counts to 10,000 per cell and applying log1p transformation

# This step is crucial for making the data comparable across cells and for stabilizing variance, 
# which is important for downstream analyses like PCA.

# The `normalize_total` function scales the counts in each cell so that they sum up to the 
# specified `target_sum` (10,000 in this case).

# The `log1p` function applies a log transformation to the data, which helps to reduce the
# skewness of the data and makes it more suitable for PCA.

# Note: The order of these operations is important. Normalization should be done before 
# log transformation.

# After this step, the data will be in a form that is more suitable for PCA and other 
# downstream analyses.

print(f"Normalizing total counts to {target_reads} per cell and applying log1p...")
sc.pp.normalize_total(adata, target_sum=target_reads)
sc.pp.log1p(adata)

Normalizing total counts to 10000.0 per cell and applying log1p...


In [14]:
# Feature Selection

# Identifying highly variable genes is a common step in single-cell RNA-seq analysis.
# It helps to focus on the most informative genes that contribute to the biological variability 
# in the data

# The `highly_variable_genes` function identifies genes that have high variability across cells,
# which are often more biologically relevant. The parameters `min_mean`, `max_mean`, 
# and `min_disp` are used to filter genes based on their mean expression and dispersion.

# The `n_top_genes` parameter specifies how many of the top variable genes to keep for 
# downstream analysis.

print(f"Identifying top {n_top_genes} highly variable genes...")
sc.pp.highly_variable_genes(
    adata, 
    min_mean=0.0125, 
    max_mean=3, 
    min_disp=0.5, 
    n_top_genes=n_top_genes
)

Identifying top 2000 highly variable genes...


In [15]:
# Save raw data state and filter

# After identifying highly variable genes, we save the raw data state in `adata.raw` before 
# subsetting the data to only include these genes. 
# This allows us to retain the original data for any future analyses that may require it.

adata.raw = adata
adata = adata[:, adata.var.highly_variable]

# Regress out technical artifacts and scale

# Regressing out technical variations such as total counts and percentage of mitochondrial genes
# helps to remove unwanted sources of variation that can confound the biological signal in the data.

print("Regressing out technical variations and scaling data...")
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)

Regressing out technical variations and scaling data...


d:\Anaconda\envs\comp_bio\Lib\site-packages\scanpy\preprocessing\_simple.py:728: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [16]:
# Run PCA and save to output file

# PCA is a dimensionality reduction technique that helps to capture the most important sources of
# variation in the data. It transforms the data into a new coordinate system where the first
# few dimensions (principal components) capture the most variance in the data. This is often used
# as a preprocessing step before clustering or visualization.

# The `svd_solver='arpack'` parameter specifies the algorithm to use for computing the PCA.
# The 'arpack' solver is efficient for large datasets and is commonly used in single-cell analysis.

print("Running Principal Component Analysis (PCA)...")
sc.tl.pca(adata, svd_solver='arpack')

print(f"Saving normalized and PCA-computed data to {output_file}...")
adata.write(output_file)
print("Normalization and PCA complete.")

Running Principal Component Analysis (PCA)...
Saving normalized and PCA-computed data to ../data/processed/pca_computed.h5ad...
Normalization and PCA complete.
